<a href="https://colab.research.google.com/github/zhaoyingjun/Tiny-R2/blob/main/Tiny_R2_journey.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tiktoken datasets transformers huggingface_hub
!pip install git+https://github.com/KellerJordan/Muon
!pip install --upgrade transformers


In [ ]:
!pip install faiss-cpu sentence_transformers datasets rouge-score nltk rank_bm25 tqdm

In [ ]:
!hf auth login --force

/bin/bash: line 1: hf: command not found


In [3]:
!git clone https://github.com/zhaoyingjun/Tiny-R2.git
%cd Tiny-R2

Cloning into 'Tiny-R2'...
remote: Enumerating objects: 530, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 530 (delta 54), reused 9 (delta 9), pack-reused 445 (from 1)
Receiving objects: 100% (530/530), 4.01 MiB | 41.43 MiB/s, done.
Resolving deltas: 100% (313/313), done.
/content/Tiny-R2


**训练climbmix数据，可以训练一个基础大模型，1.7B参数量，评测模型结构和效果。支持采用大模型作为Agent进行观察训练过程并给出实施的lr、clip的调整参数;--use_agent_observe=True开启,默认是关闭的，需要输入gemini的api_key**

默认不开启Agent智能化自主训练

In [ ]:
!python train.py --n_layer 6 --n_embd 1536 --hc 'True' --mhc 'True' --n_experts 8 --max_iters 10000 --attention_types 'Sparse' --batch_size 8 --ctx_len 2048 --hf_dataset 'karpathy/climbmix-400b-shuffle' --resume True --save_best_only True

开启Agent智能化自主训练，需要将your gemini api key替换

In [ ]:
!python train.py --n_layer 6 --n_embd 1536 --hc 'True' --mhc 'True' --n_experts 8 --max_iters 10000 --attention_types 'Sparse' --batch_size 8 --ctx_len 2048 --hf_dataset 'karpathy/climbmix-400b-shuffle' --resume True --save_best_only True --use_agent_observe True --gemini_api_key 'your gemini api key'

预训练完成之后，进行benchmark的验证

In [ ]:
!python /content/Tiny-R2/evaluate.py --checkpoint /content/Tiny-R2/checkpoints/best_model_step_1180.pt

后训练直接使用Qwen3.5-9B模型作为教师模型进行OPD训练，可自定义使用RAG增加教师模型以及自定义数据集等

In [ ]:
!python opd_train.py --hf_teacher_model Qwen/Qwen3.5-9B \
  --student_model_name tiny-r2 \
  --student_ckpt checkpoints/best_model_step_40.pt \
  --batch_size 2 \
  --grad_accum_steps 4 \
  --custom_qa_path baoxianqa.jsonl \
  --rag_corpus_path baoxianqa.txt

使用Qwen3.5-9B模型作为教师模型、Qwen3.5-0.8B作为学生模型进行OPD训练，用RAG增加教师模型，RAG数据集来自问答数据集集medquad，可以复现Readme中的结果;去除命令行中--enable_reg_teacher则关闭外挂RAG功能；--rag_corpus_path外挂RAG数据集，--custom_qa_path 自定义问题集

In [ ]:
!python opd_train.py \
  --hf_teacher_model Qwen/Qwen3.5-9B \
  --student_model_name Qwen/Qwen3.5-0.8B \
  --batch_size 2 \
  --grad_accum_steps 4\
  --student_use_rag


In [ ]:
!python opd_train_utral.py \
  --hf_teacher_model Qwen/Qwen3.5-9B \
  --student_model_name Qwen/Qwen3.5-0.8B \
  --batch_size 1 \
  --grad_accum_steps 1\
  --student_use_rag\
  --enable_wandb

默认使用medquade数据集进行OPD的结果验证，可以根据OPD训练的数据集对应的修改验证

In [ ]:
!python eval_opd.py \
    --dataset medquad \
    --hf_teacher_model Qwen/Qwen3.5-9B \
    --student_base_model Qwen/Qwen3.5-0.8B \
    --student_opd_model opd_checkpoints/student_best_model_step_450.pt \
    --eval_samples 100